# SG-1 DC Project  Task 2: Temporal Sequence Learning (CNN + LSTM)

**Architecture:** Frozen CNN backbone (Task 1) → LSTM → MLP head → 31-class gesture classification



In [ ]:
# ── Cell 1: Imports & Config ─────────────────────────────────────────────────

import os, random, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

CFG = {
    # Paths
    "data_root"     : "/kaggle/input/datasets/stellarsyntax/asl-dynamic/ASL_dynamic",
    "cnn_weights"   : "/kaggle/input/datasets/stellarsyntax/asl-cnn-weights/asl_cnn_weights (4).pth",  # adjust if needed
    "save_path"     : "asl_lstm_weights.pth",
    "frames_cache": "/kaggle/working/extracted_frames",

    # Frame / sequence
    "img_size"      : 128,       # must match Task 1
    "seq_len"       : 16,        # frames per sequence (fixed by task spec)

    # Model
    "feature_dim"   : 512,       # CNN output dim (fixed by Task 1)
    "lstm_hidden"   : 512,
    "lstm_layers"   : 2,
    "lstm_dropout"  : 0.4,
    "fc_dropout"    : 0.4,

    # Training
    "batch_size"    : 16,        # each sample is 16 frames → keep batch small
    "num_workers"   : 2,
    "epochs"        : 80,
    "lr"            : 3e-4,
    "weight_decay"  : 1e-4,
    "val_split"     : 0.20,      # 20% val (dataset is small: 31×10 = 310 videos)
    "patience"      : 15,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Seq len: {CFG['seq_len']} frames per sample")

In [ ]:
# ── Cell 2: Dataset Exploration + Frame Extraction ───────────────────────────

import cv2

IMG_EXT  = {".jpg", ".jpeg", ".png", ".bmp"}
VID_EXT  = {".avi", ".mp4", ".mov", ".mkv"}
CACHE    = Path(CFG["frames_cache"])
CACHE.mkdir(parents=True, exist_ok=True)

data_root = Path(CFG["data_root"])


def find_dynamic_root(root: Path) -> Path:
    for p in [root] + sorted(root.rglob("*")):
        if p.is_dir() and sum(1 for s in p.iterdir() if s.is_dir()) >= 26:
            return p
    raise FileNotFoundError(f"Cannot locate class root under {root}")


def extract_video_frames(video_path: Path, out_dir: Path) -> list:
    """
    Extract every frame of a .avi into out_dir as frame_0000.jpg ...
    Skips if already extracted (safe to re-run).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted([f for f in out_dir.iterdir() if f.suffix == ".jpg"])
    if existing:
        return existing  # already done

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  WARNING: cannot open {video_path.name}")
        return []

    paths, idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        out_path = out_dir / f"frame_{idx:04d}.jpg"
        cv2.imwrite(str(out_path), frame)
        paths.append(out_path)
        idx += 1
    cap.release()
    return sorted(paths)


def get_video_groups(class_dir: Path, class_name: str) -> list:
    """
    Layout A (e.g. P): frame subfolders exist → use JPGs directly
    Layout B (e.g. S): only .avi files → extract frames to CACHE first
    """
    subdirs = sorted([d for d in class_dir.iterdir() if d.is_dir()])
    videos  = sorted([f for f in class_dir.iterdir() if f.suffix.lower() in VID_EXT])

    groups = []

    if subdirs:
        # Layout A
        for sd in subdirs:
            frames = sorted([f for f in sd.iterdir() if f.suffix.lower() in IMG_EXT])
            if frames:
                groups.append(frames)

    if not groups and videos:
        # Layout B — extract frames from .avi files
        print(f"  [{class_name}] No frame folders — extracting from {len(videos)} .avi files...")
        for vid in videos:
            out_dir = CACHE / class_name / vid.stem
            frames  = extract_video_frames(vid, out_dir)
            if frames:
                groups.append(frames)

    return groups


# ── Find class root and build full inventory ──
class_root = find_dynamic_root(data_root)
print(f"Class root : {class_root}\n")

class_dirs = sorted([d for d in class_root.iterdir() if d.is_dir()])
classes    = [d.name for d in class_dirs]
print(f"Classes ({len(classes)}): {classes}\n")

print(f"{'Class':<14} {'Layout':>14} {'Videos':>7} {'Frames/vid (min-max)':>22} {'Total':>8}")
print("-" * 70)

class_video_map = {}
for cd in class_dirs:
    groups = get_video_groups(cd, cd.name)
    class_video_map[cd.name] = groups

    has_subdirs = any(d.is_dir() for d in cd.iterdir())
    layout = "frames" if has_subdirs else "avi→extracted"

    if groups:
        lengths = [len(g) for g in groups]
        print(f"{cd.name:<14} {layout:>14} {len(groups):>5}  "
              f"{min(lengths):>8}-{max(lengths):<8} {sum(lengths):>8}")
    else:
        print(f"{cd.name:<14} {'NO DATA FOUND':>40}")

print(f"\nTotal videos loaded: {sum(len(v) for v in class_video_map.values())}")

In [ ]:
# ── Cell 3: Dataset Class ─────────────────────────────────────────────────────
#
# Each sample = one video = a sequence of exactly seq_len=16 frames.
#
# Frame sampling strategy:
#   If video has >= seq_len frames → sample uniformly (covers full motion arc)
#   If video has < seq_len frames  → repeat/tile to reach seq_len (handles short clips)
#
# Train split uses mild temporal augmentation:
#   - random start offset within the uniform sampling grid (temporal jitter)
#   - random horizontal flip applied consistently to all frames in a sequence

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

frame_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


def sample_frames(frame_paths, seq_len, jitter=False):
    """
    Returns exactly seq_len frame paths sampled uniformly from frame_paths.
    jitter=True adds ±1 random offset per index (for training augmentation).
    """
    n = len(frame_paths)
    if n == 0:
        raise ValueError("Empty frame list")

    if n >= seq_len:
        indices = np.linspace(0, n - 1, seq_len)
        if jitter:
            jitter_offsets = np.random.randint(-1, 2, size=seq_len)   # -1, 0, or +1
            indices = indices + jitter_offsets
            indices = np.clip(indices, 0, n - 1)
        indices = indices.astype(int)
    else:
        # Tile then sample — handles very short clips
        tiled   = (frame_paths * (seq_len // n + 1))[:seq_len]
        indices = list(range(seq_len))
        return [tiled[i] for i in indices]

    return [frame_paths[i] for i in indices]


class ASLDynamicDataset(Dataset):
    """
    Each item: (tensor of shape [seq_len, 3, H, W], int label)

    The CNN backbone will process each of the seq_len frames independently
    to produce (seq_len, 512) feature tensors, which the LSTM then consumes.
    """

    def __init__(self, samples, class2idx, seq_len=16, train=True):
        """
        samples   : list of (frame_path_list, class_name)
        class2idx : dict mapping class name → int index
        """
        self.samples   = samples
        self.class2idx = class2idx
        self.seq_len   = seq_len
        self.train     = train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_paths, class_name = self.samples[idx]
        label = self.class2idx[class_name]

        chosen = sample_frames(frame_paths, self.seq_len, jitter=self.train)

        # Consistent horizontal flip for the whole sequence
        do_flip = self.train and (random.random() < 0.3)

        frames = []
        for fp in chosen:
            img = Image.open(fp).convert("RGB")
            if do_flip:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            frames.append(frame_tf(img))

        # Stack → (seq_len, C, H, W)
        return torch.stack(frames), label


# ── Build class → index mapping ──
# Filter out any class with 0 videos
valid_classes = [c for c in classes if len(class_video_map[c]) > 0]
print(f"Valid classes ({len(valid_classes)}): {valid_classes}")

class2idx = {c: i for i, c in enumerate(sorted(valid_classes))}
idx2class = {v: k for k, v in class2idx.items()}
NUM_CLASSES = len(class2idx)
print(f"Num classes: {NUM_CLASSES}")

# ── Flatten into (frame_list, class_name) pairs, one per video ──
all_samples = []
for cls, video_groups in class_video_map.items():
    if cls not in class2idx:
        continue
    for frame_list in video_groups:
        if len(frame_list) > 0:
            all_samples.append((frame_list, cls))

print(f"\nTotal video samples: {len(all_samples)}")
print(f"Distribution:")
from collections import Counter
dist = Counter(s[1] for s in all_samples)
for cls in sorted(dist): print(f"  {cls}: {dist[cls]}")

# ── Train / val split — split at the video level, not frame level ──
# Group by class first so each class is proportionally represented in val
by_class = defaultdict(list)
for s in all_samples:
    by_class[s[1]].append(s)

train_samples, val_samples = [], []
for cls, slist in by_class.items():
    random.shuffle(slist)
    n_val = max(1, int(len(slist) * CFG["val_split"]))  # at least 1 val per class
    val_samples.extend(slist[:n_val])
    train_samples.extend(slist[n_val:])

print(f"\nTrain videos: {len(train_samples)} | Val videos: {len(val_samples)}")

train_ds = ASLDynamicDataset(train_samples, class2idx, CFG["seq_len"], train=True)
val_ds   = ASLDynamicDataset(val_samples,   class2idx, CFG["seq_len"], train=False)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  num_workers=CFG["num_workers"],
                          pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"],
                          pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# Sanity-check a single item shape
seq, lbl = train_ds[0]
print(f"\nSingle sample shape: {seq.shape}  label: {lbl} ({idx2class[lbl]})")

In [ ]:
# ── Cell 4: Visualise a Sample Sequence ──────────────────────────────────────

def denorm(t):
    m = torch.tensor(MEAN).view(3,1,1)
    s = torch.tensor(STD).view(3,1,1)
    return torch.clamp(t * s + m, 0, 1)

# Pick a few classes and show their frame sequences
show_classes = list(class2idx.keys())[:4]
fig, axes = plt.subplots(len(show_classes), CFG["seq_len"],
                          figsize=(CFG["seq_len"] * 1.5, len(show_classes) * 1.8))

for row, cls in enumerate(show_classes):
    # Find a sample from this class in the val set
    idx = next(i for i, s in enumerate(val_samples) if s[1] == cls)
    seq, lbl = val_ds[idx]
    for col in range(CFG["seq_len"]):
        ax = axes[row][col]
        ax.imshow(denorm(seq[col]).permute(1,2,0).numpy())
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(cls, fontsize=10, rotation=0, labelpad=40, va="center")

plt.suptitle(f"Sample sequences — {CFG['seq_len']} frames each", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_sequences.png", dpi=100)
plt.show()
print("Sequence shape:", seq.shape)   # (16, 3, 128, 128)

In [ ]:
# ── Cell 5: CNN Backbone (copy from Task 1, load frozen weights) ──────────────


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4)),
            nn.ReLU(inplace=True),
            nn.Linear(max(channels // reduction, 4), channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, pool=False):
        super().__init__()
        pad       = kernel // 2
        self.conv = nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x)), inplace=True))


class ResBlock(nn.Module):
    def __init__(self, channels, se=True):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels, kernel=3)
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels)
        )
        self.se = SEBlock(channels) if se else nn.Identity()

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.se(out)
        return F.relu(out + x, inplace=True)


class ASL_CNN(nn.Module):
    """Task 1 backbone — identical to training code. Only forward_features() is used here."""

    def __init__(self, num_classes=26, feature_dim=512, dropout=0.4):
        super().__init__()
        self.stage0 = ConvBlock(3, 32, kernel=5, pool=True)
        self.stage1 = nn.Sequential(
            ConvBlock(32, 64, kernel=3, pool=True),
            ResBlock(64, se=True), ResBlock(64, se=True),
            nn.Dropout2d(0.10),
        )
        self.stage2 = nn.Sequential(
            ConvBlock(64, 128, kernel=3, pool=True),
            ResBlock(128, se=True), ResBlock(128, se=True),
            nn.Conv2d(128, 128, kernel_size=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Dropout2d(0.15),
        )
        self.stage3 = nn.Sequential(
            ConvBlock(128, 256, kernel=3, pool=True),
            ResBlock(256, se=True), nn.Dropout2d(0.20),
        )
        self.stage4 = ConvBlock(256, feature_dim, kernel=3, pool=True)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5), nn.Linear(256, num_classes),
        )
        self.feature_dim = feature_dim

    def forward_features(self, x):
        x = self.stage0(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        return self.gap(x).flatten(1)    # (B, 512)

    def forward(self, x):
        return self.head(self.forward_features(x))


# ── Load weights and freeze ──
cnn_backbone = ASL_CNN(num_classes=26, feature_dim=CFG["feature_dim"]).to(DEVICE)

ckpt = torch.load(CFG["cnn_weights"], map_location=DEVICE)
cnn_backbone.load_state_dict(ckpt["model_state"])
print(f"CNN weights loaded from epoch {ckpt['epoch']} | Task-1 val_acc={ckpt['val_acc']*100:.2f}%")

# Freeze every parameter — CNN is a feature extractor only
for param in cnn_backbone.parameters():
    param.requires_grad = False

cnn_backbone.eval()   # keep BN/Dropout in eval mode throughout

frozen = sum(p.numel() for p in cnn_backbone.parameters())
print(f"CNN params : {frozen/1e6:.2f}M — ALL FROZEN (requires_grad=False)")

# Quick shape check
with torch.no_grad():
    dummy = torch.randn(4, 3, CFG["img_size"], CFG["img_size"]).to(DEVICE)
    feat  = cnn_backbone.forward_features(dummy)
    print(f"CNN feature output: {feat.shape}")   # (4, 512)

In [ ]:

# ── Cell 6: LSTM Model ────────────────────────────────────────────────────────

class ASL_LSTM(nn.Module):

    def __init__(self, cnn, feature_dim, hidden_size, num_layers,
                 num_classes, lstm_dropout=0.4, fc_dropout=0.4):
        super().__init__()
        self.cnn         = cnn
        self.feature_dim = feature_dim
        self.hidden_size = hidden_size

        self.lstm = nn.LSTM(
            input_size   = feature_dim,
            hidden_size  = hidden_size,
            num_layers   = num_layers,
            batch_first  = True,
            dropout      = lstm_dropout if num_layers > 1 else 0.0,
            bidirectional= False,
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(
            nn.Dropout(fc_dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(fc_dropout * 0.5),
            nn.Linear(256, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for name, p in self.lstm.named_parameters():
            if "weight_ih" in name:   nn.init.xavier_uniform_(p.data)
            elif "weight_hh" in name: nn.init.orthogonal_(p.data)
            elif "bias" in name:
                p.data.zero_()
                n = p.size(0)
                p.data[n//4 : n//2].fill_(1.0)  # forget gate bias = 1
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        B, T, C, H, W = x.shape
        with torch.no_grad():
            feats = self.cnn.forward_features(x.view(B*T, C, H, W))  # (B*T, 512)
        feats = feats.view(B, T, self.feature_dim)                    # (B, T, 512)
        lstm_out, _ = self.lstm(feats)                                # (B, T, hidden)
        last = self.layer_norm(lstm_out[:, -1, :])                    # (B, hidden)
        return self.head(last)                                        # (B, num_classes)


model = ASL_LSTM(
    cnn         = cnn_backbone,
    feature_dim = CFG["feature_dim"],
    hidden_size = CFG["lstm_hidden"],
    num_layers  = CFG["lstm_layers"],
    num_classes = NUM_CLASSES,
    lstm_dropout= CFG["lstm_dropout"],
    fc_dropout  = CFG["fc_dropout"],
).to(DEVICE)

total      = sum(p.numel() for p in model.parameters())
frozen_cnt = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params        : {total/1e6:.2f}M")
print(f"Frozen (CNN)        : {frozen_cnt/1e6:.2f}M  requires_grad=False ✓")
print(f"Trainable (LSTM+FC) : {trainable/1e6:.2f}M  requires_grad=True  ✓")

# Correct freeze check: verify by requires_grad, NOT by size comparison
assert frozen_cnt > 0, "No frozen params — CNN was not frozen!"
assert trainable  > 0, "No trainable params — LSTM/head not initialized!"
assert all(not p.requires_grad for p in model.cnn.parameters()), \
    "CNN params inside model still have requires_grad=True!"
print("\nAll assertions passed ✓")

# Shape test
dummy = torch.randn(2, CFG["seq_len"], 3, CFG["img_size"], CFG["img_size"]).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape: {out.shape}")   # (2, NUM_CLASSES)




total      = sum(p.numel() for p in model.parameters())
frozen_cnt = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params        : {total/1e6:.2f}M")
print(f"Frozen (CNN)        : {frozen_cnt/1e6:.2f}M  requires_grad=False ✓")
print(f"Trainable (LSTM+FC) : {trainable/1e6:.2f}M  requires_grad=True  ✓")

# Correct freeze check: verify by requires_grad, NOT by size comparison
assert frozen_cnt > 0, "No frozen params — CNN was not frozen!"
assert trainable  > 0, "No trainable params — LSTM/head not initialized!"
assert all(not p.requires_grad for p in model.cnn.parameters()), \
    "CNN params inside model still have requires_grad=True!"
print("\nAll assertions passed ✓")

# Shape test
dummy = torch.randn(2, CFG["seq_len"], 3, CFG["img_size"], CFG["img_size"]).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape: {out.shape}")   # (2, NUM_CLASSES)


In [ ]:
# ── Cell 7: Loss, Optimiser & Scheduler ──────────────────────────────────────



trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Params given to optimizer: {sum(p.numel() for p in trainable_params)/1e6:.2f}M")

# Label smoothing: dataset is small (310 videos), smoothing reduces overconfidence
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    trainable_params,
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

# ReduceLROnPlateau works better than cosine here because the dataset is tiny
# and val loss can be noisy — we want to react to actual plateaus.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=6, min_lr=1e-6
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

print(f"Loss      : CrossEntropyLoss (label_smoothing=0.1)")
print(f"Optimizer : AdamW  lr={CFG['lr']}  wd={CFG['weight_decay']}")
print(f"Scheduler : ReduceLROnPlateau (patience=6)")

In [ ]:
# ── Cell 8: Training Loop ─────────────────────────────────────────────────────

def accuracy(logits, labels):
    return (logits.argmax(1) == labels).float().mean().item()


def run_epoch(model, loader, criterion, optimizer, scaler, train=True):
    if train:
        model.train()
        model.cnn.eval()   # CNN MUST stay in eval mode — BN/Dropout frozen
    else:
        model.eval()

    total_loss, total_acc, n = 0.0, 0.0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for seqs, labels in loader:
            seqs   = seqs.to(DEVICE, non_blocking=True)     # (B, T, C, H, W)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = model(seqs)
                loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            bs          = seqs.size(0)
            total_loss += loss.item() * bs
            total_acc  += accuracy(logits, labels) * bs
            n          += bs

    return total_loss / n, total_acc / n


history      = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
patience_ctr = 0

print(f"{'Epoch':>6} | {'T-Loss':>8} | {'T-Acc':>7} | {'V-Loss':>8} | {'V-Acc':>7} | {'LR':>9}")
print("-" * 62)

for epoch in range(1, CFG["epochs"] + 1):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
    v_loss, v_acc = run_epoch(model, val_loader,   criterion, optimizer, scaler, train=False)

    scheduler.step(v_acc)   # ReduceLROnPlateau monitors val accuracy
    lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_acc"].append(t_acc)
    history["val_acc"].append(v_acc)

    marker = ""
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        patience_ctr = 0
        torch.save({
            "epoch"       : epoch,
            "model_state" : model.state_dict(),
            "optimizer"   : optimizer.state_dict(),
            "val_acc"     : v_acc,
            "cfg"         : CFG,
            "class2idx"   : class2idx,
            "idx2class"   : idx2class,
            "num_classes" : NUM_CLASSES,
        }, CFG["save_path"])
        marker = " ← best"
    else:
        patience_ctr += 1

    print(f"{epoch:>6} | {t_loss:>8.4f} | {t_acc*100:>6.2f}% | "
          f"{v_loss:>8.4f} | {v_acc*100:>6.2f}%{marker} | {lr:.2e}")

    if patience_ctr >= CFG["patience"]:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest Val Accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# ── Cell 9: Training Curves ───────────────────────────────────────────────────

epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_ran, history["train_loss"], label="Train", linewidth=2)
axes[0].plot(epochs_ran, history["val_loss"],   label="Val",   linewidth=2)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, [a*100 for a in history["train_acc"]], label="Train", linewidth=2)
axes[1].plot(epochs_ran, [a*100 for a in history["val_acc"]],   label="Val",   linewidth=2)
axes[1].set_title("Accuracy (%)")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("ASL CNN+LSTM — Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves_lstm.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 10: Confusion Matrix & Classification Report ─────────────────────────

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Reload best checkpoint
ckpt = torch.load(CFG["save_path"], map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
model.cnn.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for seqs, labels in val_loader:
        preds = model(seqs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

class_names = [idx2class[i] for i in range(NUM_CLASSES)]
cm = confusion_matrix(all_labels, all_preds)

fig_size = max(10, NUM_CLASSES // 2)
plt.figure(figsize=(fig_size, fig_size - 2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.4, linecolor="lightgray")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix — Val Acc: {best_val_acc*100:.1f}%")
plt.tight_layout()
plt.savefig("confusion_matrix_lstm.png", dpi=150)
plt.show()

print("\nPer-class report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# ── Cell 11: Verify Freezing — Sanity Check ───────────────────────────────────
#
# Confirm CNN weights didn't change during LSTM training.
# Load original Task-1 checkpoint and compare CNN state dicts.

task1_ckpt  = torch.load(CFG["cnn_weights"], map_location="cpu")
task2_ckpt  = torch.load(CFG["save_path"],   map_location="cpu")

task1_state = task1_ckpt["model_state"]
task2_state = task2_ckpt["model_state"]

# CNN layers in task2 are stored as cnn.stage0.conv.weight etc.
mismatches = []
for key, val in task1_state.items():
    task2_key = f"cnn.{key}"
    if task2_key in task2_state:
        if not torch.allclose(val.float(), task2_state[task2_key].float()):
            mismatches.append(key)

if mismatches:
    print(f"WARNING: {len(mismatches)} CNN layers changed during training!")
    for m in mismatches:
        print(f"  {m}")
else:
    print("✓ CNN weights are IDENTICAL to Task 1 — freezing worked correctly.")

print(f"\nTask 2 checkpoint keys: {list(task2_ckpt.keys())}")
print(f"class2idx: {task2_ckpt['class2idx']}")

In [ ]:
# ── Cell 12: Summary ──────────────────────────────────────────────────────────

print("=" * 55)
print("Task 2 Complete — Saved artefacts:")
print(f"  LSTM weights       → {CFG['save_path']}")
print(f"  Training curves    → training_curves_lstm.png")
print(f"  Confusion matrix   → confusion_matrix_lstm.png")
print("=" * 55)
print(f"\nBest Val Accuracy : {best_val_acc*100:.2f}%")
print(f"Checkpoint epoch  : {ckpt['epoch']}")
print(f"Num classes       : {NUM_CLASSES}")
print(f"Classes           : {class_names}")
print("\nDone. Push asl_lstm_weights.pth to your branch for Task 3.")